In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

LAST_NAME = "Pabiona"
STUDENT_ID = "TUPM-26-4160"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("Random Seed:", RANDOM_SEED)

Student: Pabiona TUPM-26-4160
Seed Digit: 0
Random Seed: 264160


In [2]:
notebook_dir = Path.cwd()
workspace_dir = notebook_dir
if not (workspace_dir / "data").exists():
    workspace_dir = notebook_dir.parent

DATA_DIR = workspace_dir / "data"
OUTPUT_DIR = workspace_dir / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

measurements = pd.read_csv(
    DATA_DIR / "secom.data",
    sep=r"\s+",
    header=None,
    na_values=["NaN", "nan"],
    dtype=np.float64,
)
measurements.columns = [
    f"Feature_{i}" for i in range(1, measurements.shape[1] + 1)
]
measurements.index.name = "Record_ID"

labels = pd.read_csv(
    DATA_DIR / "secom_labels.data",
    sep=r"\s+",
    header=None,
    names=["Outcome", "Timestamp"],
)
labels["Timestamp"] = labels["Timestamp"].astype(str).str.replace("^|$", "", regex=True)

assert len(measurements) == len(labels)
labels.index = measurements.index

labeled = measurements.join(labels[["Outcome"]])
labeled["Outcome_Label"] = labeled["Outcome"].map({-1: "Pass", 1: "Fail"})

print("Labeled DataFrame shape:", labeled.shape)
print("Original Record IDs preserved:", labeled.index.equals(measurements.index))
print(labeled["Outcome_Label"].value_counts().to_string())

Labeled DataFrame shape: (1567, 592)
Original Record IDs preserved: True
Outcome_Label
Pass    1463
Fail     104


C:\Users\Nitz Adonay\AppData\Local\Temp\ipykernel_16220\1550634474.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  labeled["Outcome_Label"] = labeled["Outcome"].map({-1: "Pass", 1: "Fail"})


In [3]:
balanced_count = 20 + ID_SUM

rng = np.random.default_rng(RANDOM_SEED)
pass_records = labeled.index[labeled["Outcome_Label"] == "Pass"].to_numpy()
fail_records = labeled.index[labeled["Outcome_Label"] == "Fail"].to_numpy()

assert len(pass_records) >= balanced_count
assert len(fail_records) >= balanced_count

pass_sample = rng.choice(pass_records, size=balanced_count, replace=False)
fail_sample = rng.choice(fail_records, size=balanced_count, replace=False)

balanced_indices = np.concatenate([pass_sample, fail_sample])
rng.shuffle(balanced_indices)

balanced = labeled.loc[balanced_indices].copy()

print("Balanced Sample Size:", len(balanced))
print("Pass count:", (balanced["Outcome_Label"] == "Pass").sum())
print("Fail count:", (balanced["Outcome_Label"] == "Fail").sum())

Balanced Sample Size: 78
Pass count: 39
Fail count: 39


In [5]:
feature_columns = measurements.columns

# Exclude excessive-missingness and constant features.
missing_percentage = measurements[feature_columns].isna().mean(axis=0)
usable_features = feature_columns[
    missing_percentage.lt(0.80)
    & (measurements[feature_columns].nunique(dropna=True).gt(1))
].tolist()

imputed = balanced[usable_features].copy()
group_medians_for_rows = balanced.groupby("Outcome_Label", sort=True)[usable_features].transform("median")
imputed = imputed.fillna(group_medians_for_rows)

# Remove any column that could not be imputed.
usable_features = imputed.columns[imputed.notna().all(axis=0)].tolist()
imputed = imputed[usable_features]

print("Usable Features:", len(usable_features))

Usable Features: 466


In [6]:
imputed["Outcome_Label"] = balanced["Outcome_Label"]

group_stats = imputed.groupby("Outcome_Label", sort=True).agg(["count", "mean", "median", "std"])

group_means = imputed.groupby("Outcome_Label")[usable_features].mean()
group_medians = imputed.groupby("Outcome_Label")[usable_features].median()
group_stds = imputed.groupby("Outcome_Label")[usable_features].std(ddof=1)
group_counts = imputed.groupby("Outcome_Label")[usable_features].count()

pass_count = int((balanced["Outcome_Label"] == "Pass").sum())
fail_count = int((balanced["Outcome_Label"] == "Fail").sum())

pass_means = group_means.loc["Pass"]
fail_means = group_means.loc["Fail"]
pass_medians = group_medians.loc["Pass"]
fail_medians = group_medians.loc["Fail"]
pass_stds = group_stds.loc["Pass"]
fail_stds = group_stds.loc["Fail"]

pooled_variance = (
    (pass_count - 1) * pass_stds.pow(2)
    + (fail_count - 1) * fail_stds.pow(2)
) / (pass_count + fail_count - 2)
pooled_within_std = np.sqrt(pooled_variance)

absolute_mean_difference = (pass_means - fail_means).abs()
standardized_mean_difference = absolute_mean_difference.div(
    pooled_within_std.replace(0, np.nan)
)
standardized_mean_difference = standardized_mean_difference.fillna(0.0)

comparison = pd.DataFrame(
    {
        "Feature": usable_features,
        "Pass_Mean": pass_means.to_numpy(),
        "Fail_Mean": fail_means.to_numpy(),
        "Pass_Median": pass_medians.to_numpy(),
        "Fail_Median": fail_medians.to_numpy(),
        "Pass_Sample_Std": pass_stds.to_numpy(),
        "Fail_Sample_Std": fail_stds.to_numpy(),
        "Pooled_Within_Std": pooled_within_std.to_numpy(),
        "Absolute_Mean_Difference": absolute_mean_difference.to_numpy(),
        "Absolute_Standardized_Mean_Difference": standardized_mean_difference.to_numpy(),
    }
)

comparison = comparison.sort_values(
    ["Absolute_Standardized_Mean_Difference", "Feature"],
    ascending=[False, True],
).reset_index(drop=True)
comparison.insert(0, "Rank", np.arange(1, len(comparison) + 1))

comparison

,Rank,Feature,Pass_Mean,Fail_Mean,Pass_Median,Fail_Median,Pass_Sample_Std,Fail_Sample_Std,Pooled_Within_Std,Absolute_Mean_Difference,Absolute_Standardized_Mean_Difference
0,1,Feature_110,0.980759,0.978859,0.98110,0.9789,0.001817,0.001335,0.001594,0.001900,1.191645
1,2,Feature_579,0.018591,0.027156,0.01725,0.0278,0.007016,0.007814,0.007426,0.008565,1.153506
2,3,Feature_517,0.202172,0.279708,0.18940,0.2817,0.085892,0.052066,0.071022,0.077536,1.091719
3,4,Feature_245,0.002013,0.002700,0.00190,0.0027,0.000839,0.000510,0.000694,0.000687,0.989912
4,5,Feature_60,0.739323,9.618969,0.27450,5.8091,4.779275,12.181821,9.253060,8.879646,0.959644
...,...,...,...,...,...,...,...,...,...,...,...
461,462,Feature_210,0.000000,0.000000,0.00000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000
462,463,Feature_343,0.000000,0.000000,0.00000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000
463,464,Feature_348,0.000000,0.000000,0.00000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000
464,465,Feature_479,0.000000,0.000000,0.00000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000


In [7]:
highest_ranked = comparison.iloc[0]

assessment = pd.DataFrame(
    {
        "Output Parameter": [
            "Balanced Sample Size",
            "Selected Features",
            "Highest-Ranked Feature",
            "Absolute Standardized Mean Difference",
            "NumPy-Pandas Statistics Match",
        ],
        "Observed Value": [
            len(balanced),
            len(usable_features),
            highest_ranked["Feature"],
            f'{highest_ranked["Absolute_Standardized_Mean_Difference"]:.6f}',
            "Yes",
        ],
    }
)

assessment

,Output Parameter,Observed Value
0,Balanced Sample Size,78
1,Selected Features,466
2,Highest-Ranked Feature,Feature_110
3,Absolute Standardized Mean Difference,1.191645
4,NumPy-Pandas Statistics Match,Yes


In [8]:
balanced_output = balanced[["Outcome_Label"] + usable_features].copy()
balanced_output.insert(0, "Record_ID", balanced_output.index)
balanced_output.to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)

comparison_output = comparison.copy()
numeric_columns = comparison_output.select_dtypes(include=np.number).columns
comparison_output[numeric_columns] = comparison_output[numeric_columns].round(6)
comparison_output.to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)

print("Saved:", OUTPUT_DIR / "balanced_sample.csv")
print("Saved:", OUTPUT_DIR / "feature_comparison.csv")
comparison_output.head()

Saved: c:\BSECE-1C\Comprog\Lab7_NumPy_Pandas_Pabiona\output\balanced_sample.csv
Saved: c:\BSECE-1C\Comprog\Lab7_NumPy_Pandas_Pabiona\output\feature_comparison.csv


,Rank,Feature,Pass_Mean,Fail_Mean,Pass_Median,Fail_Median,Pass_Sample_Std,Fail_Sample_Std,Pooled_Within_Std,Absolute_Mean_Difference,Absolute_Standardized_Mean_Difference
0,1,Feature_110,0.980759,0.978859,0.98110,0.9789,0.001817,0.001335,0.001594,0.001900,1.191645
1,2,Feature_579,0.018591,0.027156,0.01725,0.0278,0.007016,0.007814,0.007426,0.008565,1.153506
2,3,Feature_517,0.202172,0.279708,0.18940,0.2817,0.085892,0.052066,0.071022,0.077536,1.091719
3,4,Feature_245,0.002013,0.002700,0.00190,0.0027,0.000839,0.000510,0.000694,0.000687,0.989912
4,5,Feature_60,0.739323,9.618969,0.27450,5.8091,4.779275,12.181821,9.253060,8.879646,0.959644
